# Test Semantic Cache for Intent Router

This notebook demonstrates the semantic caching feature that checks eval dataset before using LLM.

In [1]:
from sahiloan_chatbot.application.chat_service.workflow.nodes import Nodes
from sahiloan_chatbot.application.chat_service.workflow.state import ChatState
import json
from pathlib import Path

/Users/vishnum/Library/Caches/pypoetry/virtualenvs/sahiloan-chatbot-h6twZ8gO-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-26 17:35:39.487 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:43 - Looking for eval dataset at: /Users/vishnum/sahiloan-customer-chatbot/data/evals/intent_router.json
2026-01-26 17:35:39.491 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:56 - Loading 25 eval queries for intent caching...
2026-01-26 17:35:50.700 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:68 - ✅ Cached 25 intent router embeddings


## 1. Initialize Nodes (Loads Eval Dataset)

In [2]:
print("Initializing Nodes with semantic cache...")
nodes = Nodes()

print(f"✅ Loaded {len(nodes.eval_queries)} eval queries for caching")
print(f"✅ Generated {len(nodes.eval_embeddings)} embeddings")

Initializing Nodes with semantic cache...
✅ Loaded 25 eval queries for caching
✅ Generated 25 embeddings


## 2. View Eval Dataset

In [ ]:
# Show first 10 eval queries
print("="*80)
print("EVAL DATASET SAMPLES")
print("="*80)

for i in range(min(10, len(nodes.eval_queries))):
    query = nodes.eval_queries[i]
    route = nodes.eval_routes[i]
    print(f"{i+1}. Query: {query}")
    print(f"   Route: {route}")
    print()

## 3. Test Exact Matches (Should Use Cache)

In [ ]:
# Test with exact queries from eval dataset
exact_queries = [
    "What is a home loan and how does it work in India?",
    "How many EMIs have I already paid for my SBI home loan?",
    "Here is the EC document. Let me know if there are any legal issues.",
]

print("="*80)
print("TESTING EXACT MATCHES (Should hit cache)")
print("="*80)

for query in exact_queries:
    print(f"\n🔍 Query: {query}")
    print("-"*80)
    
    state = ChatState(messages=[{"role": "user", "content": query}])
    result = nodes.intent_router(state)
    
    print(f"✅ Route: {result['route_to']}")

## 4. Test Similar Queries (Should Use Cache)

In [ ]:
# Test with similar queries (rephrased)
similar_queries = [
    "Can you explain what a home loan is and how it functions in India?",  # Similar to eval
    "How many EMI payments have I made for my home loan with SBI?",  # Similar to eval
    "I'm uploading my sale deed document. Please verify if it's valid for home loan.",  # Similar to eval
]

print("="*80)
print("TESTING SIMILAR QUERIES (Should hit cache if similarity > 0.8)")
print("="*80)

for query in similar_queries:
    print(f"\n🔍 Query: {query}")
    print("-"*80)
    
    state = ChatState(messages=[{"role": "user", "content": query}])
    result = nodes.intent_router(state)
    
    print(f"✅ Route: {result['route_to']}")

## 5. Test New Queries (Should Use LLM)

In [ ]:
# Test with completely new queries (not in eval)
new_queries = [
    "Tell me about Sahiloan's service fees",
    "What are the latest interest rates?",
    "I need help with my loan application status",
]

print("="*80)
print("TESTING NEW QUERIES (Should fall back to LLM)")
print("="*80)

for query in new_queries:
    print(f"\n🔍 Query: {query}")
    print("-"*80)
    
    state = ChatState(messages=[{"role": "user", "content": query}])
    result = nodes.intent_router(state)
    
    print(f"✅ Route: {result['route_to']}")

## 6. Manual Similarity Check

In [ ]:
# Manually check similarity scores
import numpy as np

test_query = "What types of loans does Sahiloan provide?"

print(f"Test Query: {test_query}\n")
print("="*80)
print("TOP 5 SIMILAR QUERIES FROM EVAL DATASET")
print("="*80)

# Get embedding for test query
query_embedding = nodes.embeddings.embed_query(test_query)
query_vector = np.array(query_embedding)

# Calculate similarities
similarities = []
for idx, eval_embedding in enumerate(nodes.eval_embeddings):
    eval_vector = np.array(eval_embedding)
    similarity = np.dot(query_vector, eval_vector) / (
        np.linalg.norm(query_vector) * np.linalg.norm(eval_vector)
    )
    similarities.append((similarity, idx))

# Sort by similarity
similarities.sort(reverse=True)

# Show top 5
for i, (sim, idx) in enumerate(similarities[:5], 1):
    print(f"\n{i}. Similarity: {sim:.4f}")
    print(f"   Query: {nodes.eval_queries[idx]}")
    print(f"   Route: {nodes.eval_routes[idx]}")
    print(f"   Will use: {'✅ Yes (cache)' if sim >= 0.8 else '❌ No (LLM)'}")

## 7. Performance Comparison

In [ ]:
# Compare latency: Cache vs LLM
import time

# Query that should hit cache
cached_query = "What is a home loan and how does it work in India?"

# Query that should use LLM
new_query = "Can you explain the loan process timeline?"

print("="*80)
print("PERFORMANCE COMPARISON")
print("="*80)

# Test cached query
print(f"\n1. Query (Should hit cache): {cached_query}")
start = time.time()
state = ChatState(messages=[{"role": "user", "content": cached_query}])
result = nodes.intent_router(state)
cache_time = (time.time() - start) * 1000
print(f"   Time: {cache_time:.2f}ms")
print(f"   Route: {result['route_to']}")

# Test new query
print(f"\n2. Query (Should use LLM): {new_query}")
start = time.time()
state = ChatState(messages=[{"role": "user", "content": new_query}])
result = nodes.intent_router(state)
llm_time = (time.time() - start) * 1000
print(f"   Time: {llm_time:.2f}ms")
print(f"   Route: {result['route_to']}")

# Summary
print("\n" + "="*80)
print(f"💡 Cache is {llm_time/cache_time:.1f}x faster than LLM!")
print("="*80)

## 8. Test Auto-Caching (Learning Feature)

In [ ]:
# Test the auto-caching feature
# New queries should be saved to eval dataset after LLM routing

new_unique_query = "What are the processing fees for home loans?"

print("="*80)
print("TEST: AUTO-CACHING NEW INTENTS")
print("="*80)

# Check initial count
initial_count = len(nodes.eval_queries)
print(f"\nInitial cached queries: {initial_count}")

# First call - should use LLM and save to cache
print(f"\n1️⃣ First Call (Should use LLM and save):")
print(f"   Query: {new_unique_query}")
state = ChatState(messages=[{"role": "user", "content": new_unique_query}])
result = nodes.intent_router(state)
print(f"   Route: {result['route_to']}")

# Check if it was added
after_count = len(nodes.eval_queries)
print(f"\n   Cached queries after: {after_count}")
print(f"   New entries added: {after_count - initial_count}")

# Second call - should hit cache now
print(f"\n2️⃣ Second Call (Should hit cache):")
print(f"   Query: {new_unique_query}")
state = ChatState(messages=[{"role": "user", "content": new_unique_query}])
result = nodes.intent_router(state)
print(f"   Route: {result['route_to']}")

print("\n" + "="*80)
print("💡 The query is now cached for future use!")
print("="*80)

## 9. Verify Updated Eval Dataset

In [ ]:
# Check the updated eval dataset file
import json
from pathlib import Path

eval_path = Path("../data/evals/intent_router.json")

with open(eval_path, 'r') as f:
    eval_data = json.load(f)

print(f"Total entries in eval dataset: {len(eval_data)}")
print(f"\n📝 Last 5 entries (most recently added):")
print("="*80)

for i, entry in enumerate(eval_data[-5:], 1):
    print(f"{i}. Query: {entry['input']}")
    print(f"   Route: {entry['reference']}")
    print()